# Unravel live-loss investigation

Reconcile intended daily rebalances with Hyperliquid fills from **2026-06-25 onward**, when the universe increased from 20 to 40 tickers.

This notebook downloads and caches:

1. `rebalanceevents` from Azure Table Storage (positions, weights, market snapshots, proposed orders, and order updates),
2. `usertrades` from Azure Table Storage,
3. canonical fills and funding directly from Hyperliquid.

It then performs the five-stage reconciliation: reconstruct intentions, match fills, attribute costs, calculate effective bps, and break results down by ticker/liquidity/direction/size/date. Cached files live under `notebooks/data/` and are git-ignored. No credentials are written to disk.

> Coverage warning: rebalance-event telemetry was introduced on 2026-06-28. Hyperliquid fills can cover the full interval, but intended-order reconstruction may not cover June 25–27.

## 0. Environment and investigation parameters

Use the `quant-py314` kernel. The dependency check below fails early with a clear message if the environment changes.

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from azure.data.tables import TableServiceClient
from IPython.display import display

# Expected in the quant-py314 environment: pandas, pyarrow, azure-data-tables,
# requests, matplotlib, and seaborn. Imports above are the dependency check.

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')
sns.set_theme(style='whitegrid')

START = pd.Timestamp('2026-06-25T00:00:00Z')
END = pd.Timestamp.now(tz='UTC')
STRATEGY = 'UnravelDaily'
STRATEGY_KEY = STRATEGY.lower()  # Azure partition/entity values are normalized to lowercase
EXCHANGE = 'Hyperliquid'
NETWORK = 'mainnet'
RESOURCE_GROUP = 'ResourceGroup1'
FUNCTION_APP = 'yolo-funk-prod'
HYPERLIQUID_INFO_URL = 'https://api.hyperliquid.xyz/info'
FORCE_REFRESH = False

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
CACHE_DIR = NOTEBOOK_DIR / 'data' / f'unravel_loss_{START:%Y%m%d}'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Period: {START} to {END}')
print(f'Cache:  {CACHE_DIR.resolve()}')

## 1. Resolve Azure configuration safely

The existing Azure CLI login retrieves the deployed settings. Key Vault references are resolved only in memory. The storage connection string and wallet identifiers are never printed or cached.

In [ ]:
def az_json(*args: str) -> Any:
    command = ['az', *args, '--output', 'json', '--only-show-errors']
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)

def resolve_key_vault_reference(value: str) -> str:
    match = re.fullmatch(r'@Microsoft\.KeyVault\(SecretUri=(https://[^)]+)\)', value)
    if not match:
        return value
    return az_json('keyvault', 'secret', 'show', '--id', match.group(1), '--query', 'value')

settings_rows = az_json(
    'functionapp', 'config', 'appsettings', 'list',
    '--resource-group', RESOURCE_GROUP, '--name', FUNCTION_APP,
)
settings = {row['name']: row.get('value', '') for row in settings_rows}
deployed_rebalance_schedule = settings.get('Strategies__UnravelDaily__Schedule')

storage_connection_string = settings['AzureWebJobsStorage']
address_setting = settings['Strategies__UnravelDaily__Hyperliquid__Address']
vault_setting = settings.get('Strategies__UnravelDaily__Hyperliquid__VaultAddress', '')
unravel_api_key_setting = settings['Strategies__UnravelDaily__Unravel__ApiKey']
wallet_address = resolve_key_vault_reference(address_setting)
vault_address = resolve_key_vault_reference(vault_setting) if vault_setting else None
trading_address = vault_address or wallet_address
unravel_api_key = resolve_key_vault_reference(unravel_api_key_setting)

assert re.fullmatch(r'0x[0-9a-fA-F]{40}', trading_address), 'Invalid Hyperliquid trading address'
print('Azure configuration resolved; secrets remain in memory.')
print(f'Trading account: {trading_address[:6]}…{trading_address[-4:]}')
print(f'Deployed rebalance schedule: {deployed_rebalance_schedule}')

## 2. Download Azure telemetry and cache as Parquet

Parquet preserves timestamps and numeric columns, is fast to reload, and is easy to query from pandas, DuckDB, or Polars. Azure rows retain their raw payload JSON columns.

In [ ]:
def to_snake(name: str) -> str:
    return re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()

def unwrap_azure_value(value: Any) -> Any:
    # Int64 Table properties are wrapped as EntityProperty(value, edm_type).
    return value.value if hasattr(value, 'value') and hasattr(value, 'edm_type') else value

def entity_frame(entities: Iterable[dict[str, Any]]) -> pd.DataFrame:
    rows = [{to_snake(k): unwrap_azure_value(v) for k, v in dict(entity).items()} for entity in entities]
    frame = pd.DataFrame(rows)
    for col in frame.columns:
        if 'timestamp' in col or col.endswith('_utc'):
            frame[col] = pd.to_datetime(frame[col], utc=True, errors='coerce')
    return frame

def cached_parquet(name: str, loader, force: bool = FORCE_REFRESH) -> pd.DataFrame:
    path = CACHE_DIR / f'{name}.parquet'
    if path.exists() and not force:
        cached = pd.read_parquet(path)
        if not cached.empty:
            return cached
    frame = loader()
    frame.to_parquet(path, index=False)
    return frame

table_service = TableServiceClient.from_connection_string(storage_connection_string)
start_iso = START.to_pydatetime().isoformat()
end_iso = END.to_pydatetime().isoformat()

def load_rebalance_events() -> pd.DataFrame:
    table = table_service.get_table_client('rebalanceevents')
    query = f"TimestampUtc ge datetime'{start_iso}' and TimestampUtc le datetime'{end_iso}'"
    frame = entity_frame(table.query_entities(query_filter=query))
    if not frame.empty:
        frame = frame.loc[frame['strategy_name'].str.casefold().eq(STRATEGY_KEY)].copy()
        frame = frame.sort_values(['timestamp_utc', 'run_id', 'sequence']).reset_index(drop=True)
    return frame

def sanitize_table_key(value: str) -> str:
    return value.replace('/', '|').replace('\\', '|').replace('#', '_').replace('?', '_')

def load_ingested_fills() -> pd.DataFrame:
    table = table_service.get_table_client('usertrades')
    partition = sanitize_table_key(f'{STRATEGY_KEY}|{EXCHANGE}|{NETWORK}|{trading_address}')
    escaped = partition.replace("'", "''")
    query = (
        f"PartitionKey eq '{escaped}' and "
        f"TimestampUtc ge datetime'{start_iso}' and TimestampUtc le datetime'{end_iso}'"
    )
    frame = entity_frame(table.query_entities(query_filter=query))
    if not frame.empty:
        frame = frame.sort_values('timestamp_utc').reset_index(drop=True)
    return frame

events = cached_parquet('azure_rebalance_events', load_rebalance_events)
azure_fills = cached_parquet('azure_user_trades', load_ingested_fills)
print(f'Azure rebalance events: {len(events):,}')
print(f'Azure-ingested fills:   {len(azure_fills):,}')
if not events.empty:
    print(f"Event coverage: {events.timestamp_utc.min()} to {events.timestamp_utc.max()}")
if not azure_fills.empty:
    print(f"Fill coverage:  {azure_fills.timestamp_utc.min()} to {azure_fills.timestamp_utc.max()}")

## 3. Pull canonical Hyperliquid fills and funding

`userFillsByTime` returns at most 2,000 fills per request and only the latest 10,000 are available. Pagination advances from the last returned millisecond. The Azure ingestion copy is compared with this fresh source to detect ingestion gaps.

In [ ]:
session = requests.Session()
session.headers.update({'Content-Type': 'application/json', 'User-Agent': 'yolo-loss-investigation/1.0'})

def hl_post(payload: dict[str, Any]) -> Any:
    response = session.post(HYPERLIQUID_INFO_URL, json=payload, timeout=60)
    response.raise_for_status()
    return response.json()

def fetch_hl_time_range(kind: str, start: pd.Timestamp, end: pd.Timestamp) -> list[dict[str, Any]]:
    cursor = int(start.timestamp() * 1000)
    end_ms = int(end.timestamp() * 1000)
    rows: list[dict[str, Any]] = []
    seen: set[str] = set()
    while cursor <= end_ms:
        payload = {'type': kind, 'user': trading_address, 'startTime': cursor, 'endTime': end_ms}
        if kind == 'userFillsByTime':
            payload['aggregateByTime'] = False
        batch = hl_post(payload)
        if not batch:
            break
        for row in batch:
            identity = json.dumps(row, sort_keys=True, separators=(',', ':'))
            if identity not in seen:
                seen.add(identity)
                rows.append(row)
        last_time = max(int(row['time']) for row in batch)
        # A page can end in the middle of several fills sharing one millisecond.
        # Re-query that exact boundary before advancing or those fills are lost.
        boundary_payload = {'type': kind, 'user': trading_address, 'startTime': last_time, 'endTime': last_time}
        if kind == 'userFillsByTime':
            boundary_payload['aggregateByTime'] = False
        for row in hl_post(boundary_payload):
            identity = json.dumps(row, sort_keys=True, separators=(',', ':'))
            if identity not in seen:
                seen.add(identity)
                rows.append(row)
        next_cursor = last_time + 1
        if next_cursor <= cursor or len(batch) < 500:
            break
        cursor = next_cursor
        time.sleep(0.15)
    return rows

def load_hl_fills() -> pd.DataFrame:
    frame = pd.DataFrame(fetch_hl_time_range('userFillsByTime', START, END))
    if frame.empty:
        return frame
    frame['timestamp_utc'] = pd.to_datetime(frame['time'], unit='ms', utc=True)
    return frame.sort_values('timestamp_utc').reset_index(drop=True)

def load_hl_funding() -> pd.DataFrame:
    frame = pd.DataFrame(fetch_hl_time_range('userFunding', START, END))
    if frame.empty:
        return frame
    frame['timestamp_utc'] = pd.to_datetime(frame['time'], unit='ms', utc=True)
    delta = pd.json_normalize(frame.pop('delta')).add_prefix('delta_')
    return pd.concat([frame.reset_index(drop=True), delta], axis=1).sort_values('timestamp_utc')

hl_fills = cached_parquet('hyperliquid_fills_complete_v2', load_hl_fills)
funding = cached_parquet('hyperliquid_funding', load_hl_funding)
print(f'Hyperliquid fills: {len(hl_fills):,}')
print(f'Funding entries:   {len(funding):,}')

def fill_identity(frame: pd.DataFrame) -> set[str]:
    if frame.empty:
        return set()
    trade_id_col = 'tid' if 'tid' in frame else 'trade_id'
    trade_ids = pd.to_numeric(frame[trade_id_col], errors='coerce').dropna().astype('int64').astype(str)
    return set(trade_ids)

hl_ids, azure_ids = fill_identity(hl_fills), fill_identity(azure_fills)
print(f'Fresh fills absent from Azure cache: {len(hl_ids - azure_ids):,}')
print(f'Azure fills absent from fresh pull:  {len(azure_ids - hl_ids):,}')

## 4. Step 1 — reconstruct intended orders and decision prices

In [ ]:
ALIASES = {'BONK': 'KBONK', 'FLOKI': 'KFLOKI', 'LUNC': 'KLUNC', 'NEIRO': 'KNEIRO', 'PEPE': 'KPEPE', 'SHIB': 'KSHIB'}

def canonical_coin(value: Any) -> str:
    symbol = str(value or '').upper().replace('-', '').replace('/', '').replace('_', '')
    if symbol.endswith('USDC'):
        symbol = symbol[:-4]
    return ALIASES.get(symbol, symbol)

def parse_payload(value: Any) -> Any:
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return None

market_rows = []
for event in events.loc[events.event_type.eq('MarketsFetched')].itertuples():
    payload = parse_payload(event.payload_json) or {}
    for asset_type, markets in payload.items():
        for market in markets or []:
            market_rows.append({
                'run_id': event.run_id, 'market_timestamp_utc': event.timestamp_utc,
                'coin': canonical_coin(market.get('name')), 'asset_type': asset_type,
                'bid': market.get('bid'), 'ask': market.get('ask'), 'mid': market.get('mid'),
                'spread': market.get('spread'),
            })
markets = pd.DataFrame(market_rows)
for col in ['bid', 'ask', 'mid', 'spread']:
    if col in markets:
        markets[col] = pd.to_numeric(markets[col], errors='coerce')

proposal_rows = []
for event in events.loc[events.event_type.eq('TradeProposed')].itertuples():
    trade = parse_payload(event.payload_json) or {}
    raw_side = trade.get('orderSide', '')
    side = {0: 'Buy', 1: 'Sell', '0': 'Buy', '1': 'Sell'}.get(raw_side, str(raw_side))
    quantity = pd.to_numeric(trade.get('absoluteAmount', trade.get('amount')), errors='coerce')
    proposal_rows.append({
        'run_id': event.run_id, 'proposed_at_utc': event.timestamp_utc,
        'coin': canonical_coin(trade.get('symbol', event.coin)),
        'symbol': trade.get('symbol', event.coin), 'side': side, 'quantity': abs(quantity),
        'signed_quantity': abs(quantity) if side.lower().startswith('buy') else -abs(quantity),
        'limit_price': pd.to_numeric(trade.get('limitPrice'), errors='coerce'),
        'post_price': pd.to_numeric(trade.get('postPrice'), errors='coerce'),
        'order_type': trade.get('orderType'), 'reduce_only': trade.get('reduceOnly'),
        'client_order_id': trade.get('clientOrderId', event.client_order_id),
    })
proposals = pd.DataFrame(proposal_rows)
if not proposals.empty and not markets.empty:
    proposals = proposals.merge(markets, on=['run_id', 'coin'], how='left')
    proposals['reference_price'] = proposals['mid'].fillna(proposals['limit_price']).fillna(proposals['post_price'])
    proposals['intended_notional'] = proposals['quantity'] * proposals['reference_price']

coverage = (events.groupby('run_id').agg(run_start=('timestamp_utc', 'min'), run_end=('timestamp_utc', 'max'), event_count=('event_type', 'size'), completed=('event_type', lambda s: s.eq('RunCompleted').any()), failed=('event_type', lambda s: s.eq('RunFailed').any())).sort_values('run_start')) if not events.empty else pd.DataFrame()
print(f'Reconstructed proposed orders: {len(proposals):,} across {proposals.run_id.nunique() if not proposals.empty else 0:,} runs')
display(coverage.head())
display(proposals.head())

## 4a. Download Unravel signal-time closing prices

Unravel's dated price is the close at the **end** of that UTC date. A rebalance shortly after midnight therefore uses the preceding calendar day's price as its signal-time reference. The API key is held only in memory; cached Parquet contains prices, dates, and tickers only.

In [ ]:
UNRAVEL_PRICE_URL = 'https://unravel.finance/api/v1/price'
TO_UNRAVEL_TICKER = {value: key for key, value in ALIASES.items()}

def load_unravel_closes() -> pd.DataFrame:
    tickers = sorted({TO_UNRAVEL_TICKER.get(coin, coin) for coin in proposals.coin.dropna().unique()})
    response = requests.get(
        UNRAVEL_PRICE_URL,
        headers={'X-API-KEY': unravel_api_key},
        params={
            'ticker': ','.join(tickers),
            'start_date': (START - pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
            'end_date': END.strftime('%Y-%m-%d'),
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    wide = pd.DataFrame(payload['data'], index=pd.to_datetime(payload['index']), columns=payload['columns'])
    wide.index.name = 'signal_date'
    long = wide.reset_index().melt(id_vars='signal_date', var_name='unravel_ticker', value_name='unravel_close')
    long['coin'] = long.unravel_ticker.map(canonical_coin)
    long['unravel_close'] = pd.to_numeric(long.unravel_close, errors='coerce')
    return long.sort_values(['signal_date', 'coin']).reset_index(drop=True)

unravel_closes = cached_parquet('unravel_closing_prices', load_unravel_closes)
# Hyperliquid's k-prefixed aliases quote 1,000 underlying tokens per contract.
unravel_closes['exchange_price_multiplier'] = np.where(unravel_closes.unravel_ticker.isin(ALIASES), 1_000.0, 1.0)
unravel_closes['unravel_exchange_close'] = unravel_closes.unravel_close * unravel_closes.exchange_price_multiplier
print(f'Unravel close observations: {len(unravel_closes):,}')
print(f"Coverage: {unravel_closes.signal_date.min():%Y-%m-%d} to {unravel_closes.signal_date.max():%Y-%m-%d}")
display(unravel_closes.head())

## 5. Step 2 — match intentions to actual fills

Client order ID is the primary key. Order ID is retained for diagnostics. Unmatched fills are surfaced rather than force-matched.

In [ ]:
fills = hl_fills.copy()
if fills.empty:
    raise RuntimeError('No Hyperliquid fills returned for the investigation period.')
fills = fills.rename(columns={'coin': 'exchange_coin', 'px': 'fill_price', 'sz': 'fill_quantity', 'fee': 'fee_usd', 'closedPnl': 'closed_pnl_usd', 'oid': 'order_id', 'cloid': 'client_order_id'})
fills['coin'] = fills['exchange_coin'].map(canonical_coin)
fills['fill_price'] = pd.to_numeric(fills['fill_price'], errors='coerce')
fills['fill_quantity'] = pd.to_numeric(fills['fill_quantity'], errors='coerce').abs()
fills['fee_usd'] = pd.to_numeric(fills.get('fee_usd', 0), errors='coerce').fillna(0)
fills['closed_pnl_usd'] = pd.to_numeric(fills.get('closed_pnl_usd', 0), errors='coerce').fillna(0)
fills['fill_notional'] = fills['fill_price'] * fills['fill_quantity']
fills['is_buy'] = fills['side'].astype(str).str.upper().isin(['B', 'BUY'])
fills['signed_fill_quantity'] = np.where(fills['is_buy'], fills['fill_quantity'], -fills['fill_quantity'])

fill_groups = (fills.dropna(subset=['client_order_id']).groupby('client_order_id', as_index=False).agg(
    first_fill_utc=('timestamp_utc', 'min'), last_fill_utc=('timestamp_utc', 'max'),
    fill_quantity=('fill_quantity', 'sum'), signed_fill_quantity=('signed_fill_quantity', 'sum'),
    fill_notional=('fill_notional', 'sum'), fee_usd=('fee_usd', 'sum'),
    closed_pnl_usd=('closed_pnl_usd', 'sum'), fill_count=('fill_quantity', 'size'),
    crossed=('crossed', 'max'), order_ids=('order_id', lambda s: ','.join(sorted(set(s.dropna().astype(str))))),
))
fill_groups['vwap'] = fill_groups['fill_notional'] / fill_groups['fill_quantity']
matched = proposals.merge(fill_groups, on='client_order_id', how='left', indicator=True)
matched['match_status'] = np.where(matched['_merge'].eq('both'), 'matched', 'unmatched_proposal')
matched['fill_ratio'] = matched['fill_quantity'] / matched['quantity']
matched['delay_seconds'] = (matched['first_fill_utc'] - matched['proposed_at_utc']).dt.total_seconds()
matched_client_ids = set(proposals.client_order_id.dropna())
unmatched_fills = fills.loc[~fills.client_order_id.isin(matched_client_ids)].copy()
print(matched.match_status.value_counts(dropna=False))
print(f'Unmatched fills: {len(unmatched_fills):,}, notional ${unmatched_fills.fill_notional.sum():,.2f}')
display(matched.loc[matched.match_status.ne('matched'), ['proposed_at_utc', 'symbol', 'side', 'quantity', 'intended_notional', 'client_order_id']].head(20))

## 6. Steps 3–4 — attribute implementation costs and calculate effective bps

Positive values are costs and negative values are benefits. Signal delay compares the preceding Unravel UTC close with the market midpoint captured by the rebalance; fill slippage compares that midpoint with fill VWAP, so the two legs do not overlap. Fee and funding amounts use Hyperliquid's reported USDC values. Unmatched intended notional is reported as exposure shortfall, not invented P&L.

In [ ]:
matched['signal_date'] = (matched.proposed_at_utc - pd.Timedelta(days=1)).dt.tz_localize(None).dt.normalize()
matched = matched.merge(
    unravel_closes[['signal_date', 'coin', 'unravel_close', 'exchange_price_multiplier', 'unravel_exchange_close']],
    on=['signal_date', 'coin'], how='left', validate='many_to_one',
)
matched['signal_delay_usd'] = np.where(
    matched['side'].astype(str).str.lower().str.startswith('buy'),
    (matched['reference_price'] - matched['unravel_exchange_close']) * matched['fill_quantity'],
    (matched['unravel_exchange_close'] - matched['reference_price']) * matched['fill_quantity'],
)
matched['slippage_usd'] = np.where(
    matched['side'].astype(str).str.lower().str.startswith('buy'),
    (matched['vwap'] - matched['reference_price']) * matched['fill_quantity'],
    (matched['reference_price'] - matched['vwap']) * matched['fill_quantity'],
)
matched['fee_bps'] = 10_000 * matched['fee_usd'] / matched['fill_notional']
matched['signal_delay_bps'] = 10_000 * matched['signal_delay_usd'] / matched['fill_notional']
matched['slippage_bps'] = 10_000 * matched['slippage_usd'] / matched['fill_notional']
matched['post_decision_cost_usd'] = matched['fee_usd'] + matched['slippage_usd']
matched['implementation_cost_usd'] = matched['post_decision_cost_usd'] + matched['signal_delay_usd']
matched['implementation_bps'] = 10_000 * matched['implementation_cost_usd'] / matched['fill_notional']

funding_usd = pd.to_numeric(funding.get('delta_usdc', pd.Series(dtype=float)), errors='coerce').fillna(0).sum()
matched_notional = matched['fill_notional'].sum(skipna=True)
fees_usd = matched['fee_usd'].sum(skipna=True)
slippage_usd = matched['slippage_usd'].sum(skipna=True)
signal_delay_usd = matched['signal_delay_usd'].sum(skipna=True)
post_decision_cost_usd = fees_usd + slippage_usd - funding_usd  # positive funding receipt reduces cost
post_decision_bps = 10_000 * post_decision_cost_usd / matched_notional if matched_notional else np.nan
known_cost_usd = post_decision_cost_usd + signal_delay_usd
effective_bps = 10_000 * known_cost_usd / matched_notional if matched_notional else np.nan
unmatched_intended_notional = matched.loc[matched.match_status.ne('matched'), 'intended_notional'].sum()

summary = pd.Series({
    'proposed_orders': len(proposals), 'matched_orders': matched.match_status.eq('matched').sum(),
    'matched_fill_notional_usd': matched_notional, 'fees_usd': fees_usd,
    'decision_price_slippage_usd': slippage_usd, 'net_funding_received_usd': funding_usd,
    'signal_to_rebalance_delay_usd': signal_delay_usd,
    'post_decision_implementation_cost_usd': post_decision_cost_usd,
    'post_decision_effective_cost_bps': post_decision_bps,
    'known_total_implementation_cost_usd': known_cost_usd, 'known_total_effective_cost_bps': effective_bps,
    'unmatched_intended_notional_usd': unmatched_intended_notional,
    'median_fill_delay_seconds': matched.delay_seconds.median(),
    'p95_fill_delay_seconds': matched.delay_seconds.quantile(.95),
})
display(summary.to_frame('value'))
print(f"Post-decision effective cost: {post_decision_bps:.2f} bps.")
print(f"Including signal-to-rebalance movement: {effective_bps:.2f} bps versus the 5 bps and fitted 50 bps backtest assumptions.")

## 7. Step 5 — break down the shortfall

The tables below isolate whether the drag clusters in particular tickers, dates, trade directions, small/large orders, wide-spread names, taker fills, or delayed executions.

In [ ]:
analysis = matched.loc[matched.match_status.eq('matched')].copy()
analysis['date'] = analysis['proposed_at_utc'].dt.date
analysis['direction'] = np.where(analysis.side.astype(str).str.lower().str.startswith('buy'), 'Buy', 'Sell')
analysis['size_bucket'] = pd.qcut(analysis['fill_notional'], q=4, labels=['Q1 smallest', 'Q2', 'Q3', 'Q4 largest'], duplicates='drop')
analysis['spread_bps'] = 10_000 * (analysis['ask'] - analysis['bid']) / analysis['mid']
analysis['spread_bucket'] = pd.qcut(analysis['spread_bps'], q=4, labels=['Q1 tightest', 'Q2', 'Q3', 'Q4 widest'], duplicates='drop')

def cost_breakdown(group_cols: str | list[str]) -> pd.DataFrame:
    result = analysis.groupby(group_cols, observed=True).agg(
        orders=('client_order_id', 'nunique'), fill_notional_usd=('fill_notional', 'sum'),
        fees_usd=('fee_usd', 'sum'), slippage_usd=('slippage_usd', 'sum'),
        signal_delay_usd=('signal_delay_usd', 'sum'),
        median_delay_s=('delay_seconds', 'median'), crossed_rate=('crossed', 'mean'),
        median_spread_bps=('spread_bps', 'median'),
    ).reset_index()
    result['post_decision_cost_usd'] = result.fees_usd + result.slippage_usd
    result['cost_usd'] = result.post_decision_cost_usd + result.signal_delay_usd
    result['effective_bps'] = 10_000 * result.cost_usd / result.fill_notional_usd
    return result.sort_values('cost_usd', ascending=False)

by_ticker = cost_breakdown('coin')
by_date = cost_breakdown('date')
by_direction = cost_breakdown('direction')
by_size = cost_breakdown('size_bucket')
by_spread = cost_breakdown('spread_bucket')

display(by_ticker.head(20))
display(by_direction)
display(by_size)
display(by_spread)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
by_ticker.head(15).sort_values('cost_usd').plot.barh(x='coin', y='cost_usd', ax=axes[0, 0], title='Known cost by ticker (top 15)')
by_date.sort_values('date').plot.line(x='date', y='cost_usd', marker='o', ax=axes[0, 1], title='Known cost by rebalance date')
sns.scatterplot(data=analysis, x='spread_bps', y='implementation_bps', hue='crossed', size='fill_notional', alpha=.65, ax=axes[1, 0])
axes[1, 0].axhline(5, color='green', ls='--', label='5 bps')
axes[1, 0].axhline(50, color='red', ls='--', label='50 bps')
axes[1, 0].set_title('Cost vs quoted spread')
analysis.set_index('proposed_at_utc')['implementation_cost_usd'].sort_index().cumsum().plot(ax=axes[1, 1], title='Cumulative known implementation cost')
plt.tight_layout()
plt.show()

## 8. Coverage, exceptions, and exported investigation tables

These exception tables are part of the result. A clean reconciliation should not hide unmatched orders, duplicate IDs, missing decision mids, incomplete runs, or the pre-telemetry interval.

In [ ]:
exceptions = {
    'unmatched_proposals': matched.loc[matched.match_status.ne('matched')],
    'unmatched_fills': unmatched_fills,
    'missing_reference_prices': matched.loc[matched.reference_price.isna()],
    'incomplete_runs': coverage.loc[~coverage.completed] if not coverage.empty else coverage,
    'duplicate_proposal_cloids': proposals.loc[proposals.client_order_id.duplicated(keep=False)] if not proposals.empty else proposals,
}
for name, frame in exceptions.items():
    print(f'{name}: {len(frame):,}')
    frame.to_parquet(CACHE_DIR / f'{name}.parquet', index=False)

matched.to_parquet(CACHE_DIR / 'order_fill_reconciliation.parquet', index=False)
by_ticker.to_parquet(CACHE_DIR / 'cost_by_ticker.parquet', index=False)
by_date.to_parquet(CACHE_DIR / 'cost_by_date.parquet', index=False)
summary.rename_axis('metric').reset_index(name='value').to_csv(CACHE_DIR / 'summary.csv', index=False)

coverage_report = pd.Series({
    'requested_start_utc': START, 'first_rebalance_event_utc': events.timestamp_utc.min() if not events.empty else pd.NaT,
    'first_hyperliquid_fill_utc': fills.timestamp_utc.min(),
    'rebalance_runs': len(coverage), 'completed_runs': coverage.completed.sum() if not coverage.empty else 0,
    'azure_fills': len(azure_fills), 'fresh_hyperliquid_fills': len(hl_fills),
    'fresh_fills_missing_from_azure': len(hl_ids - azure_ids),
})
display(coverage_report.to_frame('value'))
print(f'Investigation artifacts exported to {CACHE_DIR.resolve()}')

## 9. Live/backtest schedule parity

Production currently uses `0 10,30 0 * * *`, whereas the checked-in `appsettings.json` uses `0 10 0 * * *`. This section quantifies the additional 00:30 execution. It is a parity difference, not automatically a loss: the second run can alter exposure as well as incur costs.

In [ ]:
run_starts = events.loc[events.event_type.eq('RunStarted'), ['run_id', 'timestamp_utc']].copy()
run_starts['date'] = run_starts.timestamp_utc.dt.date
def classify_run_slot(timestamp: pd.Timestamp) -> str:
    if timestamp.hour == 0 and 5 <= timestamp.minute < 20:
        return '00:10'
    if timestamp.hour == 0 and 25 <= timestamp.minute < 40:
        return '00:30'
    return 'manual/other'

run_starts['slot'] = run_starts.timestamp_utc.map(classify_run_slot)
runs_per_day = run_starts.groupby('date').size()
print('Rebalance runs per day:')
display(runs_per_day.value_counts().sort_index().rename_axis('runs').to_frame('days'))

analysis['schedule_slot'] = analysis.proposed_at_utc.map(classify_run_slot)
by_schedule_slot = cost_breakdown('schedule_slot')
display(by_schedule_slot)
by_schedule_slot.to_parquet(CACHE_DIR / 'cost_by_schedule_slot.parquet', index=False)

second_slot = by_schedule_slot.loc[by_schedule_slot.schedule_slot.eq('00:30')]
if not second_slot.empty:
    row = second_slot.iloc[0]
    print(f"00:30 slot: ${row.fill_notional_usd:,.0f} matched turnover, ${row.post_decision_cost_usd:,.2f} post-decision cost, ${row.signal_delay_usd:,.2f} signal-delay impact.")

## 10. Execution-failure audit

An emitted `OrderError` does not necessarily mean the intended trade failed. In particular, Hyperliquid can report a cancellation error after an order has already filled. This audit joins every error to canonical fills and separates noisy order-management errors from persistent exposure shortfalls.

In [ ]:
def classify_order_error(message: Any) -> str:
    text = str(message or '').lower()
    if 'could not cancel order' in text and 'unknown error (ok)' in text:
        return 'cancel_unknown_ok'
    if 'timeout' in text:
        return 'timeout_other'
    if 'margin' in text:
        return 'margin'
    if 'price' in text:
        return 'price'
    return 'other'

error_rows = []
for event in events.loc[events.event_type.eq('OrderError')].itertuples():
    payload = parse_payload(event.payload_json) or {}
    message = payload.get('error') or payload.get('message') or event.summary
    error_rows.append({
        'run_id': event.run_id, 'timestamp_utc': event.timestamp_utc,
        'coin': canonical_coin(event.coin or payload.get('symbol')),
        'client_order_id': event.client_order_id, 'error_message': message,
        'error_class': classify_order_error(message),
    })
order_errors = pd.DataFrame(error_rows)
error_audit = order_errors.merge(
    matched[['client_order_id', 'quantity', 'fill_quantity', 'fill_ratio', 'fill_notional', 'match_status']],
    on='client_order_id', how='left', validate='many_to_one',
)
error_audit['ultimately_filled'] = error_audit.fill_quantity.fillna(0).gt(0)
error_summary = error_audit.groupby('error_class', dropna=False).agg(
    error_events=('client_order_id', 'size'), affected_orders=('client_order_id', 'nunique'),
    ultimately_filled=('ultimately_filled', 'sum'), median_fill_ratio=('fill_ratio', 'median'),
    filled_notional_usd=('fill_notional', 'sum'),
).reset_index()
display(error_summary.sort_values('error_events', ascending=False))

run_audit = events.groupby('run_id').agg(
    run_start=('timestamp_utc', 'min'), run_end=('timestamp_utc', 'max'),
    event_count=('event_type', 'size'),
    completed=('event_type', lambda s: s.eq('RunCompleted').any()),
    failed=('event_type', lambda s: s.eq('RunFailed').any()),
    order_errors=('event_type', lambda s: s.eq('OrderError').sum()),
).reset_index()
run_audit['slot'] = run_audit.run_start.map(classify_run_slot)
run_audit['duration_minutes'] = (run_audit.run_end - run_audit.run_start).dt.total_seconds() / 60
display(run_audit.groupby(['slot', 'completed', 'failed']).size().rename('runs').reset_index())

underfills = matched.loc[matched.match_status.eq('matched') & matched.fill_ratio.lt(.995)].copy()
material_underfills = underfills.loc[(underfills.quantity - underfills.fill_quantity).abs().mul(underfills.reference_price).gt(10)]
print(f'Orders filled below 99.5%: {len(underfills):,}')
print(f'Material underfills (> $10 at decision price): {len(material_underfills):,}')
display(material_underfills[['proposed_at_utc', 'coin', 'side', 'quantity', 'fill_quantity', 'fill_ratio', 'client_order_id']].head(20))
order_errors.to_parquet(CACHE_DIR / 'order_errors.parquet', index=False)
error_audit.to_parquet(CACHE_DIR / 'order_error_fill_audit.parquet', index=False)
run_audit.to_parquet(CACHE_DIR / 'run_completion_audit.parquet', index=False)
material_underfills.to_parquet(CACHE_DIR / 'material_underfills.parquet', index=False)

## 11. Classify the 00:30 safety-net trades

Each scheduled 00:30 proposal is compared with that day's 00:10 proposal for the same coin. Opposite-direction orders are reversals/corrections; same-direction orders are residual completion or resizing; an order with no 00:10 counterpart is a new safety-run trade. Target-weight equality is checked independently from the `WeightsCalculated` snapshots.

In [ ]:
proposal_compare = matched.copy()
proposal_compare['date'] = proposal_compare.proposed_at_utc.dt.date
proposal_compare['slot'] = proposal_compare.proposed_at_utc.map(classify_run_slot)
primary = proposal_compare.loc[proposal_compare.slot.eq('00:10'), ['date', 'coin', 'signed_quantity', 'quantity', 'fill_ratio']].rename(columns={
    'signed_quantity': 'primary_signed_quantity', 'quantity': 'primary_quantity', 'fill_ratio': 'primary_fill_ratio',
})
safety = proposal_compare.loc[proposal_compare.slot.eq('00:30')].merge(primary, on=['date', 'coin'], how='left')
primary_dates = set(proposal_compare.loc[proposal_compare.slot.eq('00:10'), 'date'])
same_direction = np.sign(safety.signed_quantity).eq(np.sign(safety.primary_signed_quantity))
safety['safety_class'] = np.select(
    [
        ~safety.date.isin(primary_dates),
        safety.primary_signed_quantity.isna(),
        same_direction & safety.primary_fill_ratio.lt(.995),
        same_direction,
    ],
    [
        'primary_run_missing',
        'new_after_no_primary_proposal',
        'residual_after_primary_underfill',
        'same_direction_resize_after_full_fill',
    ],
    default='opposite_direction_reversal_or_correction',
)
safety_summary = safety.groupby('safety_class').agg(
    orders=('client_order_id', 'nunique'), fill_notional_usd=('fill_notional', 'sum'),
    post_decision_cost_usd=('post_decision_cost_usd', 'sum'), signal_delay_usd=('signal_delay_usd', 'sum'),
    median_primary_fill_ratio=('primary_fill_ratio', 'median'), median_safety_fill_ratio=('fill_ratio', 'median'),
).reset_index()
display(safety_summary)

weight_rows = []
for event in events.loc[events.event_type.eq('WeightsCalculated')].itertuples():
    for ticker, weight in (parse_payload(event.payload_json) or {}).items():
        weight_rows.append({'run_id': event.run_id, 'weights_at_utc': event.timestamp_utc, 'date': event.timestamp_utc.date(), 'slot': classify_run_slot(event.timestamp_utc), 'coin': canonical_coin(ticker), 'target_weight': float(weight)})
target_weights = pd.DataFrame(weight_rows)
w10 = target_weights.loc[target_weights.slot.eq('00:10'), ['date', 'coin', 'target_weight']].rename(columns={'target_weight': 'target_0010'})
w30 = target_weights.loc[target_weights.slot.eq('00:30'), ['date', 'coin', 'target_weight']].rename(columns={'target_weight': 'target_0030'})
weight_parity = w10.merge(w30, on=['date', 'coin'], how='outer')
weight_parity['target_change_abs'] = (weight_parity.target_0030 - weight_parity.target_0010).abs()
print(f"Target rows differing by >1e-10 between 00:10 and 00:30: {(weight_parity.target_change_abs > 1e-10).sum():,} / {len(weight_parity):,}")
display(weight_parity.loc[weight_parity.target_change_abs.gt(1e-10)].head(20))
safety.to_parquet(CACHE_DIR / 'safety_run_classification.parquet', index=False)
safety_summary.to_parquet(CACHE_DIR / 'safety_run_summary.parquet', index=False)
weight_parity.to_parquet(CACHE_DIR / '0010_0030_target_weight_parity.parquet', index=False)

## 12. Reconstruct and validate the live position ledger

The first `PositionsFetched` snapshot anchors the ledger. Every later position is independently reconstructed by applying all Hyperliquid fills after the anchor—not only fills matched to Yolo proposals. Differences are valued using that run's captured market midpoint. This detects missing fills, external/manual activity not present in the extract, and position-state inconsistencies.

In [ ]:
position_rows = []
for event in events.loc[events.event_type.eq('PositionsFetched')].itertuples():
    payload = parse_payload(event.payload_json) or {}
    for ticker, positions_for_ticker in payload.items():
        for position in positions_for_ticker or []:
            position_rows.append({
                'run_id': event.run_id, 'snapshot_utc': event.timestamp_utc,
                'slot': classify_run_slot(event.timestamp_utc), 'coin': canonical_coin(position.get('baseAsset', ticker)),
                'reported_quantity': float(position.get('amount', 0)),
            })
position_snapshots = pd.DataFrame(position_rows).sort_values(['snapshot_utc', 'coin']).reset_index(drop=True)
anchor_utc = position_snapshots.snapshot_utc.min()
anchor = position_snapshots.loc[position_snapshots.snapshot_utc.eq(anchor_utc)].set_index('coin').reported_quantity.to_dict()

ledger_fills = fills.loc[fills.timestamp_utc.gt(anchor_utc), ['timestamp_utc', 'coin', 'signed_fill_quantity']].copy()
coins = sorted(set(position_snapshots.coin) | set(ledger_fills.coin))
reconstructed = []
for snapshot_utc in position_snapshots.snapshot_utc.drop_duplicates().sort_values():
    cumulative = ledger_fills.loc[ledger_fills.timestamp_utc.le(snapshot_utc)].groupby('coin').signed_fill_quantity.sum()
    for coin in coins:
        reconstructed.append({
            'snapshot_utc': snapshot_utc, 'coin': coin,
            'reconstructed_quantity': anchor.get(coin, 0.0) + cumulative.get(coin, 0.0),
        })
reconstructed_positions = pd.DataFrame(reconstructed)
position_validation = position_snapshots.merge(reconstructed_positions, on=['snapshot_utc', 'coin'], how='outer')
position_validation['reported_quantity'] = position_validation.reported_quantity.fillna(0)
position_validation['quantity_difference'] = position_validation.reported_quantity - position_validation.reconstructed_quantity
position_validation = position_validation.merge(markets[['run_id', 'coin', 'mid']], on=['run_id', 'coin'], how='left')
position_validation['difference_usd'] = position_validation.quantity_difference * position_validation.mid
position_validation['absolute_difference_usd'] = position_validation.difference_usd.abs()

snapshot_validation = position_validation.groupby(['run_id', 'snapshot_utc', 'slot'], dropna=False).agg(
    coins=('coin', 'size'), mismatched_coins=('absolute_difference_usd', lambda s: s.gt(5).sum()),
    gross_absolute_difference_usd=('absolute_difference_usd', 'sum'),
    max_absolute_difference_usd=('absolute_difference_usd', 'max'),
).reset_index()
print(f'Ledger anchor: {anchor_utc}')
print(f"Snapshots within $5 gross reconstruction error: {(snapshot_validation.gross_absolute_difference_usd <= 5).sum():,} / {len(snapshot_validation):,}")
display(snapshot_validation.sort_values('gross_absolute_difference_usd', ascending=False).head(20))
display(position_validation.sort_values('absolute_difference_usd', ascending=False)[['snapshot_utc', 'slot', 'coin', 'reported_quantity', 'reconstructed_quantity', 'quantity_difference', 'mid', 'difference_usd']].head(30))
position_snapshots.to_parquet(CACHE_DIR / 'reported_position_snapshots.parquet', index=False)
reconstructed_positions.to_parquet(CACHE_DIR / 'reconstructed_position_ledger.parquet', index=False)
position_validation.to_parquet(CACHE_DIR / 'position_ledger_validation.parquet', index=False)
snapshot_validation.to_parquet(CACHE_DIR / 'position_snapshot_validation_summary.parquet', index=False)

## 13. Compare exported backtest holdings with live target weights

The supplied unsmoothed CSV with at least 120 days of pre-period price history is treated as the authoritative backtest holdings export. Backtest weights are normalized to unit gross exposure before comparison because the export's gross exposure varies, whereas each live `WeightsCalculated` payload sums to unit gross exposure. Missing weights are treated as zero positions. Exported signal date *t* is aligned to the production rebalance on *t+1*, because live telemetry confirms that the rebalance uses the prior UTC day's Unravel factors.

**Configuration finding:** this replacement export uses the synchronized eight-factor set without the backtest's former blanket five-day rolling mean. Production enum `Carry` maps to the Unravel API identifier `carry_enhanced`; the other seven factors also agree. The deployed application has no `Smoothing` override, so the production API request leaves smoothing unset.

In [ ]:
BACKTEST_WEIGHTS_CSV = Path(os.environ.get(
    'BACKTEST_WEIGHTS_CSV',
    Path.home() / 'Downloads' / 'unravel_backtest_portfolio_weights_2026-09-11.csv',
))
if not BACKTEST_WEIGHTS_CSV.exists():
    raise FileNotFoundError(f'Set BACKTEST_WEIGHTS_CSV to the exported holdings CSV: {BACKTEST_WEIGHTS_CSV}')

backtest_weights_raw = pd.read_csv(BACKTEST_WEIGHTS_CSV, index_col=0)
backtest_weights_raw.index = pd.to_datetime(backtest_weights_raw.index).normalize()
backtest_weights_raw.index.name = 'date'
backtest_gross = backtest_weights_raw.abs().sum(axis=1).replace(0, np.nan)
backtest_weights = backtest_weights_raw.div(backtest_gross, axis=0)
BACKTEST_SIGNAL_TO_REBALANCE_DAYS = 1
backtest_weights_aligned = backtest_weights.copy()
backtest_weights_aligned.index = backtest_weights_aligned.index + pd.Timedelta(days=BACKTEST_SIGNAL_TO_REBALANCE_DAYS)
backtest_gross_aligned = backtest_gross.copy()
backtest_gross_aligned.index = backtest_gross_aligned.index + pd.Timedelta(days=BACKTEST_SIGNAL_TO_REBALANCE_DAYS)

live_targets = (target_weights.assign(
        date=pd.to_datetime(target_weights.date),
        coin=target_weights.coin.replace({'KPEPE': 'PEPE', 'KSHIB': 'SHIB'}),
    )
    .sort_values(['date', 'slot'])
    .drop_duplicates(['date', 'coin'])
    .pivot(index='date', columns='coin', values='target_weight'))
# The live strategy was intentionally reverted to 20 tickers on 2026-09-08.
# Exclude that intervention and compare only the stable 40-ticker regime.
DIAGNOSTIC_END = pd.Timestamp('2026-09-07')
common_dates = backtest_weights_aligned.index.intersection(live_targets.index)
common_dates = common_dates[common_dates <= DIAGNOSTIC_END]
all_coins = backtest_weights_aligned.columns.union(live_targets.columns)
bt = backtest_weights_aligned.reindex(index=common_dates, columns=all_coins).fillna(0)
live = live_targets.reindex(index=common_dates, columns=all_coins).fillna(0)

target_parity_daily = pd.DataFrame(index=common_dates)
target_parity_daily['backtest_assets'] = backtest_weights_aligned.reindex(common_dates).notna().sum(axis=1)
target_parity_daily['live_assets'] = live_targets.reindex(common_dates).notna().sum(axis=1)
target_parity_daily['overlapping_assets'] = ((bt != 0) & (live != 0)).sum(axis=1)
target_parity_daily['target_l1_distance'] = (bt - live).abs().sum(axis=1)
target_parity_daily['target_correlation'] = [bt.loc[d].corr(live.loc[d]) for d in common_dates]
target_parity_daily['backtest_raw_gross'] = backtest_gross_aligned.reindex(common_dates)

target_differences = (bt.stack().rename('backtest_weight').to_frame()
    .join(live.stack().rename('live_target_weight'))
    .reset_index(names=['date', 'coin']))
target_differences['absolute_difference'] = (target_differences.live_target_weight - target_differences.backtest_weight).abs()
display(target_parity_daily.groupby('live_assets').agg(
    days=('live_assets', 'size'), mean_overlap=('overlapping_assets', 'mean'),
    mean_l1_distance=('target_l1_distance', 'mean'), mean_correlation=('target_correlation', 'mean'),
))
display(target_parity_daily.tail(10))
display(target_differences.sort_values('absolute_difference', ascending=False).head(30))
backtest_weights_raw.to_parquet(CACHE_DIR / 'backtest_portfolio_weights.parquet')
target_parity_daily.reset_index().to_parquet(CACHE_DIR / 'backtest_live_target_parity_daily.parquet', index=False)
target_differences.to_parquet(CACHE_DIR / 'backtest_live_target_differences.parquet', index=False)

## 14. Estimate the return impact of target-weight divergence

This isolates **signal/portfolio-construction divergence**, not execution. Both portfolios are marked with the same Unravel close-to-close returns and unit gross exposure. Therefore the difference cannot be attributed to fees or fill slippage. Coverage is reported explicitly; a missing close contributes no return and is never silently converted to a zero price move.

In [ ]:
closing_prices = (unravel_closes
    .pivot(index='signal_date', columns='unravel_ticker', values='unravel_close'))
closing_prices.index = pd.to_datetime(closing_prices.index).normalize()
next_close_return = closing_prices.pct_change(fill_method=None).shift(-1)
price_coins = all_coins.intersection(next_close_return.columns)
returns = next_close_return.reindex(index=common_dates, columns=price_coins)
bt_priced = bt.reindex(columns=price_coins)
live_priced = live.reindex(columns=price_coins)

signal_return_bridge = pd.DataFrame(index=common_dates)
signal_return_bridge['backtest_target_return'] = (bt_priced * returns).sum(axis=1, min_count=1)
signal_return_bridge['live_target_return'] = (live_priced * returns).sum(axis=1, min_count=1)
signal_return_bridge['target_divergence_return'] = signal_return_bridge.live_target_return - signal_return_bridge.backtest_target_return
signal_return_bridge['backtest_gross_price_coverage'] = (bt_priced.abs().where(returns.notna()).sum(axis=1) / bt.abs().sum(axis=1)).fillna(0)
signal_return_bridge['live_gross_price_coverage'] = (live_priced.abs().where(returns.notna()).sum(axis=1) / live.abs().sum(axis=1)).fillna(0)
signal_return_bridge['live_assets'] = target_parity_daily.live_assets
signal_return_bridge['cumulative_backtest_target'] = (1 + signal_return_bridge.backtest_target_return.fillna(0)).cumprod() - 1
signal_return_bridge['cumulative_live_target'] = (1 + signal_return_bridge.live_target_return.fillna(0)).cumprod() - 1

signal_summary = pd.Series({
    'comparison_start': common_dates.min(),
    'comparison_end': common_dates.max(),
    'backtest_target_compound_return': (1 + signal_return_bridge.backtest_target_return.fillna(0)).prod() - 1,
    'live_target_compound_return': (1 + signal_return_bridge.live_target_return.fillna(0)).prod() - 1,
    'arithmetic_target_divergence': signal_return_bridge.target_divergence_return.sum(),
    'mean_backtest_price_coverage': signal_return_bridge.backtest_gross_price_coverage.mean(),
    'mean_live_price_coverage': signal_return_bridge.live_gross_price_coverage.mean(),
    'backtest_assets_without_prices': ','.join(sorted(set(backtest_weights.columns) - set(closing_prices.columns))),
})
display(signal_summary.to_frame('value'))
display(signal_return_bridge.sort_values('target_divergence_return').head(15))
ax = signal_return_bridge[['cumulative_backtest_target', 'cumulative_live_target']].plot(
    figsize=(12, 5), title='Same prices and gross exposure: backtest export vs live target returns')
ax.legend()
signal_return_bridge.reset_index().to_parquet(CACHE_DIR / 'signal_return_bridge.parquet', index=False)
signal_summary.rename_axis('metric').reset_index(name='value').to_csv(CACHE_DIR / 'signal_return_summary.csv', index=False)

## 15. Compare archived live factors with today's historical endpoint

`FactorsCalculated` telemetry preserves the raw factor rows received during each rebalance. This audit compares those point-in-time observations with `/portfolio/factors` retrieved later for the same factor date and ticker. It distinguishes genuine historical revisions from downstream portfolio-construction differences.

In [ ]:
factor_api_names = {
    'Carry': 'carry_enhanced', 'InstantaneousMomentum': 'instantaneous_momentum',
    'MeanReversion': 'mean_reversion', 'OpenInterestDivergence': 'open_interest_divergence',
    'RelativeIlliquidity': 'relative_illiquidity', 'RetailFlow': 'retail_flow',
    'SupplyVelocity': 'supply_velocity', 'TrendLongonlyAdaptive': 'trend_longonly_adaptive',
}
live_factor_rows = []
factor_events = events.loc[events.event_type.eq('FactorsCalculated')].sort_values('timestamp_utc').copy()
factor_events['rebalance_date'] = factor_events.timestamp_utc.dt.normalize()
for event in factor_events.drop_duplicates('rebalance_date').itertuples():
    live_factor_rows.extend((parse_payload(event.payload_json) or {}).get('rawFactors', []))
archived_raw_factors = pd.DataFrame(live_factor_rows)
archived_raw_factors['Date'] = pd.to_datetime(archived_raw_factors.Date).dt.normalize()
archived_raw_factors = archived_raw_factors.loc[archived_raw_factors.Date.le(DIAGNOSTIC_END - pd.Timedelta(days=1))]

def load_live_historical_factor_comparison() -> pd.DataFrame:
    tickers = sorted(archived_raw_factors.Ticker.unique())
    comparisons = []
    for live_column, api_name in factor_api_names.items():
        response = requests.get(
            'https://unravel.finance/api/v1/portfolio/factors',
            params={'id': api_name, 'tickers': ','.join(tickers),
                    'start_date': archived_raw_factors.Date.min().date().isoformat(),
                    'end_date': archived_raw_factors.Date.max().date().isoformat()},
            headers={'X-API-KEY': unravel_api_key}, timeout=60)
        response.raise_for_status()
        body = response.json()
        historical = pd.DataFrame(body['data'], index=pd.to_datetime(body['index']), columns=body['columns'])
        historical.index = historical.index.normalize()
        historical = historical.stack(future_stack=True).rename('historical_value').reset_index()
        historical.columns = ['Date', 'Ticker', 'historical_value']
        live_values = archived_raw_factors[['Date', 'Ticker', live_column]].rename(columns={live_column: 'live_value'})
        joined = live_values.merge(historical, on=['Date', 'Ticker'], how='left')
        joined['factor'] = api_name
        joined['difference'] = joined.historical_value - joined.live_value
        comparisons.append(joined)
    return pd.concat(comparisons, ignore_index=True)

factor_revision_audit = cached_parquet('live_vs_historical_raw_factors', load_live_historical_factor_comparison)
factor_revision_summary = factor_revision_audit.groupby('factor').agg(
    observations=('live_value', 'size'), historical_coverage=('historical_value', 'count'),
    mean_absolute_difference=('difference', lambda x: x.abs().mean()),
    max_absolute_difference=('difference', lambda x: x.abs().max()),
    materially_changed=('difference', lambda x: x.abs().gt(1e-4).sum()),
).reset_index()
display(factor_revision_summary)
display(factor_revision_audit.assign(abs_difference=lambda x: x.difference.abs())
        .nlargest(25, 'abs_difference')[['Date', 'Ticker', 'factor', 'live_value', 'historical_value', 'difference']])
factor_revision_summary.to_parquet(CACHE_DIR / 'live_vs_historical_factor_summary.parquet', index=False)

## Interpretation guardrails

- **50 bps is a fitted backtest parameter**, not proof that exchange costs equal 50 bps.
- The known-cost estimate includes signal-close-to-rebalance movement, decision-mid-to-fill slippage, reported fees, and funding. It deliberately does not invent dollar P&L for unmatched exposure.
- The supplied daily holdings now permit a target-return bridge. It identifies signal/portfolio-construction divergence, but a final account-level residual still requires a time series of live account equity or realized P&L aligned to the same checkpoints.
- Hyperliquid documents a 10,000-fill historical limit for `userFillsByTime`; the Azure-ingested table is therefore the durable source if the period eventually exceeds that limit.